In [ ]:
# v,a,dの名前がズレてないか確認(間違ってなかった！)
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"
)

print(config.id2label)

/home/mitani/.conda/envs/bunseki/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mitani/.conda/envs/bunseki/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{0: 'arousal', 1: 'dominance', 2: 'valence'}


In [ ]:
# vad-model.ipynbの成功例を抜粋
# 音声からVADを抽出し，VA/VD/ADにおける感情強度を可視化

In [ ]:
# ogvc
# 量産版（一応出た）
# 次元ベクトル（v,a,d）をOGVC_VAD_bulk.csvに保存
import os
import re
import csv
from pathlib import Path

import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn
from transformers import Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)
from tqdm import tqdm

# ===== 設定 =====
ROOT_PATH = "/autofs/diamond/share/corpus/OGVC/Vol2/Acted/wav"
OUTPUT_DIR = "/home/mitani/CSJ-emo-int_bunseki/0718/wagner_csj_vad/results"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE_RATE = 16000
MODEL_NAME = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"

# emotion dict: 3文字コード → 表示ラベル（必要に応じて拡張）
emotion_dict = {
    "JOY": "JOY", "ACC": "ACC", "FEA": "FEA", "SUR": "SUR", "SAD": "SAD",
    "DIS": "DIS", "ANG": "ANG", "ANT": "ANT", "NEU": "NEU", "OTH": "OTH"
}


# ===== モデル定義 =====
class RegressionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        x = features
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class EmotionModel(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = RegressionHead(config)
        self.init_weights()

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        hidden_states = outputs[0]
        hidden_states = torch.mean(hidden_states, dim=1)
        logits = self.classifier(hidden_states)
        return hidden_states, logits


# モデル・プロセッサのロード
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = EmotionModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()


# ===== ユーティリティ =====
def collect_wav_files(root: str):
    return sorted([str(x) for x in Path(root).rglob("*.wav")])


def parse_filename_info(basename: str):
    name = os.path.splitext(os.path.basename(basename))[0].upper()
    m = re.match(r"^([FM][A-Z]{2}\d{4})([A-Z]{3})([0-3])$", name)
    if m:
        utt_id, emo, inten = m.groups()
        return utt_id, emo if emo in emotion_dict else None, int(inten)
    return None, None, None


def load_audio(path, target_sr=SAMPLE_RATE):
    wav, sr = sf.read(path)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        wav = librosa.resample(y=wav, orig_sr=sr, target_sr=target_sr)
        sr = target_sr
    return wav.astype(np.float32), sr


def process_func(x: np.ndarray, sampling_rate: int) -> np.ndarray:
    y = processor(x, sampling_rate=sampling_rate)
    y = y['input_values'][0]
    y = y.reshape(1, -1)
    y = torch.from_numpy(y).to(DEVICE)
    with torch.no_grad():
        logits = model(y)[1]
    return logits.detach().cpu().numpy()


# ===== メイン =====
def extract_vad_bulk(root_dir=ROOT_PATH, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    wav_files = collect_wav_files(root_dir)

    records = []
    for path in tqdm(wav_files, desc="Processing WAV files"):
        utt_id, emo, inten = parse_filename_info(path)
        if utt_id is None or inten is None:
            continue
        try:
            wav, sr = load_audio(path)
            vad = process_func(wav, sr)
            records.append({
                "utt_id": utt_id,
                "filename": os.path.basename(path),
                "valence": float(vad[0][2]),
                "arousal": float(vad[0][0]),
                "dominance": float(vad[0][1]),
                "intensity": inten,
                "emotion": emo if emo else "UNK"
            })
        except Exception as e:
            print(f"Error processing {path}: {e}")

    # CSV 保存
    csv_path = os.path.join(output_dir, "OGVC_VAD_results.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as wf:
        writer = csv.writer(wf)
        writer.writerow(["utt_id", "filename", "valence", "arousal", "dominance", "intensity", "emotion"])
        for r in records:
            writer.writerow([r["utt_id"], r["filename"], r["valence"], r["arousal"], r["dominance"],
                             r["intensity"], r["emotion"]])
    print(f"Saved all results to {csv_path}")


if __name__ == "__main__":
    extract_vad_bulk()

In [5]:
# ogvcのVADをヒートマップ表示（VA/VD/AD）
import os
import pandas as pd
import matplotlib.pyplot as plt

# === 入力CSV ===
csv_path = "VA_results_bulk/OGVC_VAD_bulk.csv"
df = pd.read_csv(csv_path)

# === ペア設定（列名 → フォルダ名） ===
pair_list = [
    ("valence", "arousal", "VA"),
    ("valence", "dominance", "VD"),
    ("arousal", "dominance", "AD"),
]

# === 軸範囲（あなたのデータに合わせて変更 OK） ===
axis_min, axis_max = -0.1, 1.1

# === 各ペアごとに処理 ===
emotions = df["emotion"].unique()

for x_col, y_col, pair_name in pair_list:
    # --- 保存フォルダ作成 ---
    output_dir = f"VAD_heatmaps_ogvc/{pair_name}"
    os.makedirs(output_dir, exist_ok=True)

    print(f"▶ {pair_name} のヒートマップを保存中...")

    # --- グラフ作成 ---
    for emotion in emotions:
        subset = df[df["emotion"] == emotion]

        plt.figure(figsize=(7, 6))
        scatter = plt.scatter(
            subset[x_col],
            subset[y_col],
            c=subset["intensity"],
            cmap="viridis",
            s=20,
            alpha=0.8,
            edgecolors="k"
        )

        # カラーバー
        plt.colorbar(scatter, label="Intensity")

        # ラベル・タイトル
        plt.title(f"{pair_name}: {emotion}")
        plt.xlabel(x_col.capitalize())
        plt.ylabel(y_col.capitalize())

        # 軸範囲
        plt.xlim(axis_min, axis_max)
        plt.ylim(axis_min, axis_max)

        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()

        # 保存
        filename = f"{emotion}_{pair_name}_heatmap.png"
        plt.savefig(os.path.join(output_dir, filename))
        plt.close()

    print(f"✅ {pair_name} のヒートマップを {output_dir} に保存しました。")

print("🎉 全ての VAD ヒートマップの保存が完了しました。")

▶ VA のヒートマップを保存中...
✅ VA のヒートマップを VAD_heatmaps_ogvc/VA に保存しました。
▶ VD のヒートマップを保存中...
✅ VD のヒートマップを VAD_heatmaps_ogvc/VD に保存しました。
▶ AD のヒートマップを保存中...
✅ AD のヒートマップを VAD_heatmaps_ogvc/AD に保存しました。
🎉 全ての VAD ヒートマップの保存が完了しました。


In [ ]:
# ==============================================
# CREMA-D VAD 抽出・CSV 保存 (ファイル探索アルゴリズム更新版)　一応できた
# ==============================================

import os
import glob
import csv
import numpy as np
import soundfile as sf
import librosa
from tqdm import tqdm
import torch
import torch.nn as nn
from transformers import Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)

# =========================
# CREMA-D 設定
# =========================
ROOT_PATH = "/autofs/diamond2/share/corpus/CREMA-D/AudioWAV"
OUTPUT_DIR = "CREMA_D_VAD_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# emotion / intensity dict
emotion_dict = {"ANG":0, "DIS":1, "FEA":2, "HAP":3, "NEU":4, "SAD":5}
intensity_dict = {"LO":0, "MD":1, "HI":2}

# =========================
# EmotionModel 定義
# =========================
class RegressionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        x = self.dropout(features)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x

class EmotionModel(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = RegressionHead(config)
        self.init_weights()

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        hidden_states = outputs[0]
        hidden_states = torch.mean(hidden_states, dim=1)
        logits = self.classifier(hidden_states)
        return hidden_states, logits

# =========================
# モデルロード
# =========================
MODEL_NAME = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = EmotionModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# =========================
# 音声読み込み関数
# =========================
def load_audio(path, target_sr=16000):
    wav, sr = sf.read(path)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        wav = librosa.resample(y=wav, orig_sr=sr, target_sr=target_sr)
        sr = target_sr
    return wav.astype(np.float32), sr

# =========================
# process_func（VAD抽出）
# =========================
def process_func(x: np.ndarray, sampling_rate: int, embeddings: bool = False) -> np.ndarray:
    y = processor(x, sampling_rate=sampling_rate)
    y = y['input_values'][0].reshape(1, -1)
    y = torch.from_numpy(y).to(DEVICE)
    with torch.no_grad():
        hidden_states, logits = model(y)
        output = hidden_states if embeddings else logits
    return output.detach().cpu().numpy()

# =========================
# CREMA-D ファイル探索（新アルゴリズム）
# =========================
def collect_wav_files(root):
    return sorted(glob.glob(os.path.join(root, "*.wav")))

def parse_filename(filename):
    name = os.path.splitext(os.path.basename(filename))[0]
    parts = name.split("_")
    if len(parts) >= 4:
        speaker, sentence, emo, intensity = parts[0], parts[1], parts[2], parts[3]
        return speaker, sentence, emo, intensity
    return None, None, None, None

wav_files = collect_wav_files(ROOT_PATH)
file_infos = []

for f in wav_files:
    speaker, sentence, emo, intensity = parse_filename(f)
    if speaker is None or emo not in emotion_dict or intensity not in intensity_dict:
        continue
    file_infos.append((f, speaker, sentence, emo, intensity))

print(f"✅ CREMA-D ファイル件数: {len(file_infos)}")

# =========================
# VAD 抽出 & CSV 保存
# =========================
output_csv = os.path.join(OUTPUT_DIR, "CREMA_D_VAD_results.csv")
with open(output_csv, "w", newline="", encoding="utf-8") as wf:
    writer = csv.writer(wf)
    writer.writerow(["filename", "arousal", "dominance", "valence", "emotion", "intensity"])

    for f, speaker, sentence, emo, intensity in tqdm(file_infos, desc="Processing CREMA-D"):
        wav, sr = load_audio(f)
        vad = process_func(wav, sr)  # 元の VAD 抽出
        writer.writerow([
            os.path.basename(f),
            float(vad[0][0]),
            float(vad[0][1]),
            float(vad[0][2]),
            emo,
            intensity
        ])

print(f"✅ CREMA-D VAD 結果 CSV 保存完了: {output_csv}")

In [ ]:
# creama-d版 ヒートマップ作成（完了）(-0.1~1.1) これが一番わかりやすい！
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# CSV 読み込み
# =========================
csv_path = "CREMA_D_VAD_results/CREMA_D_VAD_results.csv"
df = pd.read_csv(csv_path)

# =========================
# 保存フォルダ作成
# =========================
plot_base_dir = "CREMA_D_VAD_results/plots_01-11"
os.makedirs(plot_base_dir, exist_ok=True)
plot_dirs = {}
for pair_name in ["VA", "VD", "AD"]:
    dir_path = os.path.join(plot_base_dir, pair_name)
    os.makedirs(dir_path, exist_ok=True)
    plot_dirs[pair_name] = dir_path

# =========================
# ヒートマップ作成関数
# =========================
def plot_scatter(df, x_col, y_col, save_dir, size_factor=50):
    for emo in df['emotion'].unique():
        plt.figure(figsize=(7,6))
        subset = df[df['emotion']==emo]
        plt.scatter(
            subset[x_col], subset[y_col],
            c=subset['intensity'].map({"LO":0, "MD":0.5, "HI":1}),
            cmap="viridis",
            s=20, edgecolors='k', alpha=0.8    # s=size_factor;50
        )
        plt.xlabel(x_col.capitalize())
        plt.ylabel(y_col.capitalize())
        plt.title(f"{y_col.capitalize()} vs {x_col.capitalize()} ({emo})")
        plt.grid(True)
        # plt.xlim(0,1)
        # plt.ylim(0,1)
        plt.xlim(-0.1,1.1)
        plt.ylim(-0.1,1.1)
        plt.colorbar(label="Intensity")
        plt.tight_layout()
        save_path = os.path.join(save_dir, f"{emo}_{y_col}_{x_col}.png")
        plt.savefig(save_path, dpi=200)
        plt.close()


# =========================
# 各ペアのプロット
# =========================
# Valence-Arousal
plot_scatter(df, "valence", "arousal", plot_dirs["VA"])
# Valence-Dominance
plot_scatter(df, "valence", "dominance", plot_dirs["VD"])
# Arousal-Dominance
plot_scatter(df, "arousal", "dominance", plot_dirs["AD"])

print(f"✅ ヒートマップ保存完了: {plot_base_dir}")
